# 2. Train one model

Reusable: set `MODEL_NAME` below, Run All, then change it and Run All
again for the next backbone. Requires notebook 1 to have been run at
least once (needs `data/train`, `data/val`, `class_names.json`,
`class_weights.pt` on Drive).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
DRIVE_ROOT = "/content/drive/MyDrive/comparison/boilerplate/"
sys.path.append(DRIVE_ROOT)

import fer_common as fc
paths = fc.get_paths(DRIVE_ROOT)


Mounted at /content/drive


In [ ]:
!ls -la /content/drive/MyDrive/comparison/boilerplate/data/

total 17
-rw------- 1 root root   67 Aug 16 21:21 'class_names (1).json'
-rw------- 1 root root   67 Aug 16 21:49  class_names.json
-rw------- 1 root root 1619 Aug 16 21:21 'class_weights (1).pt'
-rw------- 1 root root 1619 Aug 16 21:49  class_weights.pt
drwx------ 2 root root 4096 Jul  6 14:58  test
drwx------ 2 root root 4096 Aug 16 21:09  train
drwx------ 2 root root 4096 Aug 16 21:16  val


## Load class names + class weights (saved by notebook 1, on Drive)

In [ ]:
import json
import torch
from torch.utils.data import DataLoader
from torchvision import datasets

class_names = fc.load_class_names(paths)
class_weights = fc.load_class_weights(paths)
print("Classes:", class_names)


Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


## Copy train/val from Drive to local disk (once per session)

In [ ]:
import shutil

LOCAL_TRAIN_DIR = "/content/data/train"
LOCAL_VAL_DIR = "/content/data/val"

if not os.path.exists(LOCAL_TRAIN_DIR):
    print("Copying train/val from Drive to local disk...")
    shutil.copytree(paths["train_dir"], LOCAL_TRAIN_DIR)
    shutil.copytree(paths["val_dir"], LOCAL_VAL_DIR)
    print("Done.")
else:
    print("Local data already present, skipping copy.")


Copying train/val from Drive to local disk...
Done.


## Config — change `MODEL_NAME` and re-run for each backbone

In [ ]:
MODEL_NAME = "inception_v3"  # <-- one of: inception_v3 DONE, densenet121 DONE, shufflenet_v2_x2_0 DONE,
                          #     resnet18 DONE, regnet_y_400mf DONE, mobilenet_v3_small DONE,
                          #     googlenet DONE, efficientnet_v2_s DONE, alexnet DONE

BATCH_SIZE = 64
EPOCHS = 30
LEARNING_RATE = 1e-4

cfg = fc.get_model_config(MODEL_NAME)
print(f"Training {cfg['name']} at input size {cfg['input_size']}")


Training inception_v3 at input size 299


## Loaders — local files, no sampler needed since the split is already physical

In [ ]:
train_tf = fc.build_transforms(cfg["input_size"], train=True)
val_tf = fc.build_transforms(cfg["input_size"], train=False)

train_dataset = datasets.ImageFolder(LOCAL_TRAIN_DIR, transform=train_tf)
val_dataset = datasets.ImageFolder(LOCAL_VAL_DIR, transform=val_tf)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                           num_workers=4, pin_memory=True, persistent_workers=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=4, pin_memory=True, persistent_workers=True)

print(f"Train: {len(train_dataset)}  Val: {len(val_dataset)}")


Train: 22967  Val: 5742


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## Train

In [ ]:
model_weights_dir = os.path.join(paths["weights_dir"], MODEL_NAME)
os.makedirs(model_weights_dir, exist_ok=True)
ckpt_path = os.path.join(model_weights_dir, "best_model.pth")

net = fc.BackboneClassifier(cfg["fn"], num_classes=len(class_names), lr=LEARNING_RATE, epochs=EPOCHS)

history = net.train_(
    train_loader,
    epochs=EPOCHS,
    class_weights=class_weights,
    val_loader=val_loader,
    ckpt_path=ckpt_path,
)


Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth


100%|██████████| 104M/104M [00:00<00:00, 179MB/s] 
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


Epoch [1/30] Train Loss: 1.7899 | Val Loss: 1.7128 | Val Acc: 0.5491
Epoch [2/30] Train Loss: 1.5533 | Val Loss: 1.6416 | Val Acc: 0.5998
Epoch [3/30] Train Loss: 1.4532 | Val Loss: 1.6114 | Val Acc: 0.6358
Epoch [4/30] Train Loss: 1.3867 | Val Loss: 1.5939 | Val Acc: 0.6385
Epoch [5/30] Train Loss: 1.3284 | Val Loss: 1.5539 | Val Acc: 0.6614
Epoch [6/30] Train Loss: 1.2670 | Val Loss: 1.5543 | Val Acc: 0.6749
Epoch [7/30] Train Loss: 1.2155 | Val Loss: 1.5626 | Val Acc: 0.6715
Epoch [8/30] Train Loss: 1.1592 | Val Loss: 1.5496 | Val Acc: 0.6877
Epoch [9/30] Train Loss: 1.1135 | Val Loss: 1.6023 | Val Acc: 0.6661
Epoch [10/30] Train Loss: 1.0628 | Val Loss: 1.6203 | Val Acc: 0.6749
Epoch [11/30] Train Loss: 1.0258 | Val Loss: 1.6201 | Val Acc: 0.6801
Epoch [12/30] Train Loss: 0.9815 | Val Loss: 1.6380 | Val Acc: 0.6844
Epoch [13/30] Train Loss: 0.9470 | Val Loss: 1.6509 | Val Acc: 0.6773
Epoch [14/30] Train Loss: 0.9098 | Val Loss: 1.6429 | Val Acc: 0.6806
Epoch [15/30] Train Loss: 0.8

## Save training history

`best_model.pth` was already saved during training (by `train_`) — this just records the per-epoch curve alongside it.

In [ ]:
history_path = os.path.join(model_weights_dir, "history.json")
with open(history_path, "w") as f:
    json.dump(history, f)

print(f"Saved weights to {ckpt_path}")
print(f"Saved history to {history_path}")


Saved weights to /content/drive/MyDrive/comparison/boilerplate/weights/inception_v3/best_model.pth
Saved history to /content/drive/MyDrive/comparison/boilerplate/weights/inception_v3/history.json


## Next

Change `MODEL_NAME` above to the next backbone and Run All again.
Once every model you want is trained, move to notebook 3 to evaluate
all of them together.